# M-1 one-epoch smoke test

Run every cell from top to bottom in a fresh Colab session with a GPU runtime. This checks the training pipeline only; it is not E0 training. The smoke configuration exposes only train and validation paths, so the test split is not evaluated. The actual result will therefore contain validation metrics only, not `test_map50`.

The notebook checks out the exact trainer commit below. Do not add dataset archives, weights, credentials, or tokens to the repository.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Runtime > Change runtime type.")
print(torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/tairqaldy/jolscan-ict-final-1st-trimester.git"
BRANCH = "model/m1-smoke"
COMMIT = "3004fdfe8f59e7a261743ec01c390e40d811d791"
REPO = Path("/content/jolscan-m1")
if REPO.exists():
    raise FileExistsError(f"Use a fresh Colab runtime; {REPO} already exists.")
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", COMMIT], check=True)
actual_commit = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True, text=True
).stdout.strip()
if actual_commit != COMMIT:
    raise RuntimeError(f"Expected {COMMIT}, got {actual_commit}")
print(f"Checked out {actual_commit}")

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "ml/requirements.txt")],
    check=True,
)

Mount Drive when prompted. The next cell requires exactly one file named `potholes_yolo_v1.zip` under My Drive or Shared drives. It verifies and extracts the uploaded archive as-is; it does not recreate or alter the split.

In [ ]:
from google.colab import drive
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
import json
import yaml

drive.mount("/content/drive")
drive_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/Shareddrives")]
archives = sorted({p for root in drive_roots if root.exists() for p in root.rglob("potholes_yolo_v1.zip")})
if len(archives) != 1:
    raise RuntimeError(f"Expected exactly one potholes_yolo_v1.zip in mounted Drive; found {len(archives)}")

DATA_ROOT = Path("/content/data")
if DATA_ROOT.exists():
    raise FileExistsError(f"Use a fresh Colab runtime; {DATA_ROOT} already exists.")
with ZipFile(archives[0]) as archive:
    bad_member = archive.testzip()
    if bad_member is not None:
        raise RuntimeError(f"Corrupt ZIP member: {bad_member}")
    for member in archive.infolist():
        parts = PurePosixPath(member.filename).parts
        if PurePosixPath(member.filename).is_absolute() or ".." in parts:
            raise RuntimeError(f"Unsafe ZIP path: {member.filename}")
    archive.extractall(DATA_ROOT)

data_yaml = DATA_ROOT / "data.yaml"
manifest_path = DATA_ROOT / "split_manifest.json"
if not data_yaml.is_file() or not manifest_path.is_file():
    raise FileNotFoundError("The archive must contain root-level data.yaml and split_manifest.json")
with data_yaml.open(encoding="utf-8") as source:
    data_config = yaml.safe_load(source)
with manifest_path.open(encoding="utf-8") as source:
    manifest = json.load(source)
if data_config.get("names") != {0: "pothole"}:
    raise ValueError(f"Unexpected class names: {data_config.get('names')}")
expected_images = {"train": 465, "val": 100, "test": 100}
expected_objects = {"train": 1198, "val": 305, "test": 236}
if manifest.get("version") != "potholes_yolo_v1":
    raise ValueError(f"Unexpected dataset version: {manifest.get('version')}")
if manifest.get("image_counts") != expected_images or manifest.get("object_counts") != expected_objects:
    raise ValueError("Manifest counts do not match the recorded frozen split.")
for split, expected in expected_images.items():
    image_dir = DATA_ROOT / "images" / split
    image_count = sum(1 for path in image_dir.iterdir() if path.is_file() and path.suffix.lower() in {".jpg", ".jpeg"})
    if image_count != expected:
        raise ValueError(f"{split}: expected {expected} images, found {image_count}")
print("Verified frozen split metadata and image counts:", expected_images, expected_objects)

In [ ]:
import ultralytics

config_dir = Path(ultralytics.__file__).parent / "cfg" / "models" / "26"
configs = sorted(path.name for path in config_dir.glob("*.yaml"))
print("Ultralytics:", ultralytics.__version__)
print("YOLO26 configs:", configs)
required_configs = {"yolo26.yaml", "yolo26-p2.yaml"}
if not required_configs.issubset(configs):
    raise RuntimeError("Expected YOLO26 configs are missing; stop and report this before changing model families.")

In [ ]:
subprocess.run(
    [
        sys.executable,
        "ml/train.py",
        "--run", "E0",
        "--data", str(DATA_ROOT / "data.yaml"),
        "--dataset-version", "potholes_yolo_v1",
        "--smoke",
    ],
    cwd=REPO,
    check=True,
)

In [ ]:
import mlflow
import pandas as pd
from google.colab import files
from zipfile import ZIP_DEFLATED

result_path = REPO / "ml/results/runs/E0-smoke.json"
csv_path = REPO / "ml/results/runs/E0-smoke-results.csv"
result = json.loads(result_path.read_text(encoding="utf-8"))
if result["commit"] != COMMIT:
    raise RuntimeError(f"Result commit mismatch: {result['commit']}")
if result["evaluation_splits"] != ["val"] or result["test_evaluated"]:
    raise RuntimeError("Smoke run must evaluate validation only.")
if any(key.startswith("test_") for key in result["metrics"]):
    raise RuntimeError("Smoke result unexpectedly contains test metrics.")
if len(result["metrics"]) != 6 or not csv_path.is_file():
    raise RuntimeError("Expected six validation metrics and the actual training CSV.")

runs = mlflow.search_runs(experiment_names=["jolscan-pothole"])
smoke_runs = runs[runs["tags.mlflow.runName"] == "E0-smoke"]
if len(smoke_runs) != 1 or smoke_runs.iloc[0]["status"] != "FINISHED":
    raise RuntimeError(f"Expected one finished E0-smoke MLflow run; found {len(smoke_runs)}")
if "metrics.test_map50" in smoke_runs and pd.notna(smoke_runs.iloc[0]["metrics.test_map50"]):
    raise RuntimeError("Smoke run must not log test_map50.")
print(smoke_runs[["tags.mlflow.runName", "status", "metrics.val_map50"]])
print("No test metrics were evaluated or logged by this smoke run.")

collection = Path("/content/E0-smoke-results.zip")
with ZipFile(collection, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(result_path, arcname="E0-smoke.json")
    archive.write(csv_path, arcname="E0-smoke-results.csv")
files.download(str(collection))